# 01 · CUDA-Q Basics

This notebook combines the **Basics/CPU** and **Basics/GPU** examples into one place:

| Part | Topic |
|---|---|
| 1 | Targets & your first kernel |
| 2 | Gates: single-qubit, controlled, adjoint, custom, kernels calling kernels, parameterized |
| 3 | Measurement, including mid-circuit measurement |
| 4 | GHZ state on CPU and GPU |
| 5 | Expectation values & optimization |
| 6 | GPU: multi-QPU and parallel circuits |

**Kernel:** select **Python (CUDA-Q GPU)** in the top-right corner (see `00_jupyter_kernel_from_container.ipynb`). Parts 1–5 run on the CPU simulator (`qpp-cpu`); Parts 4 and 6 use the GPU.

## Part 1 · Targets and your first kernel

In [ ]:
import cudaq
import numpy as np

print("CUDA-Q version:", cudaq.__version__)
print("GPUs visible  :", cudaq.num_available_gpus())
print("Default target:", cudaq.get_target().name)

### Available targets
A **target** is the backend that executes your kernels (CPU simulator, GPU simulator, tensor-network simulator, or real QPU). `cudaq.get_targets()` lists everything this container supports.

In [ ]:
for tgt in cudaq.get_targets():
    print(f"{tgt.name:<22} {tgt.description}")

Uncomment the next line to see the full details (simulator, platform, precision, supported options) of every target.

In [ ]:
# for tgt in cudaq.get_targets(): print(tgt)

For Parts 1–5 we use the multi-threaded CPU state-vector simulator.

In [ ]:
cudaq.set_target("qpp-cpu")
print(f"Running on target {cudaq.get_target().name}")

### Your first kernel
A kernel is a Python function decorated with `@cudaq.kernel`. This one allocates `qubit_count` qubits, puts the first one in superposition with `h`, entangles the rest with CNOTs (`x.ctrl`) and measures. `cudaq.draw` shows the circuit and `cudaq.sample` runs it (1000 shots by default).

In [ ]:
@cudaq.kernel
def kernel(qubit_count: int):
    # Allocate our qubits.
    qvector = cudaq.qvector(qubit_count)
    # Place the first qubit in the superposition state.
    h(qvector[0])
    # Loop through the allocated qubits and apply controlled-X,
    # or CNOT, operations between them.
    for qubit in range(qubit_count - 1):
        x.ctrl(qvector[qubit], qvector[qubit + 1])
    # Measure the qubits.
    mz(qvector)


qubit_count = 2
print(cudaq.draw(kernel, qubit_count))
results = cudaq.sample(kernel, qubit_count)
# Should see a roughly 50/50 distribution between the |00> and
# |11> states. Example: {00: 505  11: 495}
print("Measurement distribution:" + str(results))

### Capturing a Python variable
Kernels can also read (capture) variables from the surrounding Python scope. Here `n_qubits` is captured instead of being passed as an argument.

In [ ]:
n_qubits = 2

@cudaq.kernel
def bell_kernel():
    qubits = cudaq.qvector(n_qubits)
    h(qubits[0])
    for i in range(1, n_qubits):
        x.ctrl(qubits[0], qubits[i])
    mz(qubits)


result = cudaq.sample(bell_kernel)
print(result)  # Example: { 11:500 00:500 }

## Part 2 · Gates

### Single-qubit gates
Every non-parameterized single-qubit gate applied to one qubit, then sampled.

In [ ]:
@cudaq.kernel
def kernel():
    '''
    This is our first CUDA-Q kernel.
    '''
    # Allocate a single qubit to the kernel via `qubit()`.
    qubit = cudaq.qubit()

    # Every non-parameterized single qubit gate supported by CUDA-Q.
    h(qubit)
    x(qubit)
    y(qubit)
    z(qubit)
    t(qubit)
    s(qubit)

    # Measure so that we can sample the results on our simulator.
    mz(qubit)


# Execute the kernel `shots_count` times (default 1000) and return the
# sampled distribution as a `cudaq.SampleResult` dictionary.
result = cudaq.sample(kernel)
print(result)

### Defining kernels
`cudaq.qubit` builds a register consisting of a single qubit, while `cudaq.qvector` builds a register of $N$ qubits.

In [ ]:
@cudaq.kernel
def kernel():
    A = cudaq.qubit()
    B = cudaq.qvector(3)
    C = cudaq.qvector(5)

Inputs to kernels are defined by specifying a parameter in the kernel definition along with the appropriate type. The kernel below takes an integer to define a register of N qubits.

In [ ]:
N = 2

@cudaq.kernel
def kernel(N: int):
    register = cudaq.qvector(N)

### Initializing states
There are a few ways to define an initial state for a kernel. Method 5 is particularly useful when the state of one kernel is passed into a second kernel to prepare its initial state.

1. Passing complex vectors as parameters
2. Capturing complex vectors
3. Precision-agnostic API
4. Define as CUDA-Q amplitudes
5. Pass in a state from another kernel

In [ ]:
# 1. Passing complex vectors as parameters
c = [.707 + 0j, 0 - .707j]

@cudaq.kernel
def kernel(vec: list[complex]):
    q = cudaq.qubit(vec)


# 2. Capturing complex vectors
c = [0.70710678 + 0j, 0., 0., 0.70710678]

@cudaq.kernel
def kernel():
    q = cudaq.qvector(c)


# 3. Precision-agnostic API
c = np.array([0.70710678 + 0j, 0., 0., 0.70710678], dtype=cudaq.complex())

@cudaq.kernel
def kernel():
    q = cudaq.qvector(c)


# 4. Define as CUDA-Q amplitudes
c = cudaq.amplitudes([0.70710678 + 0j, 0., 0., 0.70710678])

@cudaq.kernel
def kernel():
    q = cudaq.qvector(c)


# 5. Pass in a state from another kernel
c = [0.70710678 + 0j, 0., 0., 0.70710678]

@cudaq.kernel
def kernel_initial():
    q = cudaq.qvector(c)


state_to_pass = cudaq.get_state(kernel_initial)

@cudaq.kernel
def kernel(state: cudaq.State):
    q = cudaq.qvector(state)


kernel(state_to_pass)
print(state_to_pass)

### Applying gates
All the predefined gates in CUDA-Q are listed at https://nvidia.github.io/cuda-quantum/latest/api/default_ops.html.

Gates can be applied to all qubits in a register:

In [ ]:
@cudaq.kernel
def kernel():
    register = cudaq.qvector(10)
    h(register)

Or to individual qubits in a register:

In [ ]:
@cudaq.kernel
def kernel():
    register = cudaq.qvector(10)
    h(register[0])   # first qubit
    h(register[-1])  # last qubit

### Controlled operations
Controlled operations are available for any gate by adding `.ctrl`, followed by the control qubit and the target qubit.

In [ ]:
@cudaq.kernel
def kernel():
    register = cudaq.qvector(10)
    x.ctrl(register[0], register[1])  # CNOT gate applied with qubit 0 as control

### Multi-controlled operations
More than one qubit can be used as control: pass the control qubits as a list.

In [ ]:
@cudaq.kernel
def kernel():
    register = cudaq.qvector(10)
    x.ctrl([register[0], register[1]], register[2])  # X on qubit 2 controlled by qubits 0 and 1

You can also call a kernel as a controlled operation from inside another kernel with `cudaq.control`:

In [ ]:
@cudaq.kernel
def x_kernel(qubit: cudaq.qubit):
    x(qubit)


# A kernel that will call `x_kernel` as a controlled operation.
@cudaq.kernel
def kernel():
    control_vector = cudaq.qvector(2)
    target = cudaq.qubit()

    x(control_vector)
    x(target)
    x(control_vector[1])
    cudaq.control(x_kernel, control_vector, target)

The above is equivalent to the following Toffoli example. Both controls end in $|1\rangle$ after the X gates, so the target flips and the result is always `101`.

In [ ]:
@cudaq.kernel
def kernel():
    qvector = cudaq.qvector(3)
    x(qvector)
    x(qvector[1])
    x.ctrl([qvector[0], qvector[1]], qvector[2])
    mz(qvector)


results = cudaq.sample(kernel)
print(cudaq.draw(kernel))
print("Measurement distribution:" + str(results))   # { 101:1000 }

### Adjoint operations
The adjoint of a gate is applied by appending `.adj` to the gate.

In [ ]:
@cudaq.kernel
def kernel():
    register = cudaq.qvector(10)
    t.adj(register[0])

### Custom operations
Custom gates are registered with `cudaq.register_operation`. A one-dimensional NumPy array gives the unitary matrix, read row by row. Custom gates support `.ctrl` just like built-in gates.

In [ ]:
cudaq.register_operation("custom_x", np.array([0, 1, 1, 0]))

@cudaq.kernel
def kernel():
    qvector = cudaq.qvector(2)
    h(qvector[0])
    custom_x(qvector[0])
    custom_x.ctrl(qvector[0], qvector[1])
    mz(qvector)


results = cudaq.sample(kernel)
print(cudaq.draw(kernel))
# Should see a roughly 50/50 distribution between |00> and |11>
print("Measurement distribution:" + str(results))

### Building kernels with kernels
For larger applications a kernel can call another kernel as a subroutine. Here `kernel_A` performs a CNOT and is called 5 times from the outer kernel.

In [ ]:
@cudaq.kernel
def kernel_A(qubit_0: cudaq.qubit, qubit_1: cudaq.qubit):
    x.ctrl(qubit_0, qubit_1)


@cudaq.kernel
def kernel(qubit_count: int):
    qvector = cudaq.qvector(qubit_count)
    for i in range(5):
        kernel_A(qvector[i], qvector[i + 1])


print(cudaq.draw(kernel, 10))

### Parameterized kernels
Parameterized kernels are the building block for variational algorithms such as VQE.

In [ ]:
@cudaq.kernel
def kernel(thetas: list[float]):
    qubits = cudaq.qvector(2)
    rx(thetas[0], qubits[0])
    ry(thetas[1], qubits[1])


thetas = [.024, .543]
kernel(thetas)
print(cudaq.draw(kernel, thetas))

## Part 3 · Measurement

### Measuring in different bases
`mz` measures in the Z basis, `mx` in the X basis (drawn as an `h` followed by a Z measurement) and `my` in the Y basis.

In [ ]:
@cudaq.kernel
def kernel():
    qubits = cudaq.qvector(2)
    mx(qubits)


print(cudaq.draw(kernel))
print(cudaq.sample(kernel))

### Mid-circuit measurement and conditional logic
A measurement result can be stored in a variable and used to control later gates. Here qubit 1 gets an `h` only if qubit 0 was measured as 1.

> **Note:** the reference run of this script on `qpp-cpu` (`cq_31447296.out`) stopped with `RuntimeError: qpp::reset(): Object has zero size!`. The cell catches that error so the rest of the notebook keeps running; switch to the GPU target (`cudaq.set_target("nvidia")`) if you hit it.

In [ ]:
@cudaq.kernel
def kernel():
    q = cudaq.qvector(2)
    h(q[0])
    b0 = mz(q[0])
    reset(q[0])
    x(q[0])

    if b0:
        h(q[1])


print(cudaq.draw(kernel))
try:
    print(cudaq.sample(kernel))
except RuntimeError as err:
    print("RuntimeError:", err)

## Part 4 · GHZ state on CPU and GPU
A GHZ state $(|00\dots0\rangle + |11\dots1\rangle)/\sqrt{2}$ is a good benchmark: the state vector doubles in size with every qubit. This example uses the **kernel-builder** API (`cudaq.make_kernel`), an alternative to the `@cudaq.kernel` decorator.

We choose the backend with `cudaq.set_target`: `nvidia` for a single GPU or `qpp-cpu` for the multi-threaded CPU.

In [ ]:
import timeit

def ghz_state(N):
    kernel = cudaq.make_kernel()
    q = kernel.qalloc(N)
    kernel.h(q[0])
    for i in range(N - 1):
        kernel.cx(q[i], q[i + 1])
    kernel.mz(q)
    return kernel

**GPU** (`nvidia` target, single GPU). 28 qubits is 2 GiB of state vector in single precision.

In [ ]:
cudaq.set_target("nvidia")

n = 28
print("Preparing GHZ state for", n, "qubits on target", cudaq.get_target().name)
ghz = ghz_state(n)
start = timeit.default_timer()
counts = cudaq.sample(ghz)
print(f"time: {timeit.default_timer() - start:.2f} s")
counts.dump()

**CPU** (`qpp-cpu`). The reference CPU run of 28 qubits took **~76 s on a full 128-core RM node** (`cq_31447293.out`). A Jupyter session on GPU-shared has far fewer cores, so this cell uses 24 qubits to keep the wait short. Raise `n` to 28 to compare directly with the GPU.

In [ ]:
cudaq.set_target("qpp-cpu")

n = 24
print("Preparing GHZ state for", n, "qubits on target", cudaq.get_target().name)
ghz = ghz_state(n)
start = timeit.default_timer()
counts = cudaq.sample(ghz)
print(f"time: {timeit.default_timer() - start:.2f} s")
counts.dump()

## Part 5 · Expectation values and optimization
`cudaq.observe` computes $\langle\psi(\theta)|H|\psi(\theta)\rangle$ for a spin Hamiltonian built from `cudaq.spin` operators. Here we minimise the energy of a 2-qubit (deuteron) Hamiltonian with the **Adam** optimizer and **central-difference** gradients.

In [ ]:
from cudaq import spin

cudaq.set_target("qpp-cpu")

hamiltonian = 5.907 - 2.1433 * spin.x(0) * spin.x(1) - 2.1433 * spin.y(
    0) * spin.y(1) + .21829 * spin.z(0) - 6.125 * spin.z(1)


@cudaq.kernel
def ansatz(angles: list[float]):
    qubits = cudaq.qvector(2)
    x(qubits[0])
    ry(angles[0], qubits[1])
    x.ctrl(qubits[1], qubits[0])


optimizer = cudaq.optimizers.Adam()
gradient = cudaq.gradients.CentralDifference()


def objective_function(parameter_vector: list[float],
                       hamiltonian=hamiltonian,
                       gradient_strategy=gradient,
                       kernel=ansatz) -> tuple[float, list[float]]:

    get_result = lambda parameter_vector: cudaq.observe(
        kernel, hamiltonian, parameter_vector).expectation()

    cost = get_result(parameter_vector)
    gradient_vector = gradient_strategy.compute(parameter_vector, get_result, cost)
    return cost, gradient_vector


energy, parameter = optimizer.optimize(dimensions=1, function=objective_function)

print(f"minimized <H> = {round(energy, 16)}")      # reference: -1.7483830311526454
print(f"optimal theta 0 = {round(parameter[0], 16)}")  # reference:  0.5840908448487905

## Part 6 · GPU: multi-QPU and parallel circuits

The `nvidia` target with `option="mqpu"` treats **each GPU as a separate virtual QPU**, so independent work (Hamiltonian terms, parameter sets, circuits) can run on several GPUs at the same time.

> With the session requested in the video (`--gres=gpu:h100-80:1`) you have **1 GPU = 1 QPU**, so the code runs but there is nothing to split across. Request 2+ GPUs (e.g. `--gres=gpu:h100-80:2`) to see the speedup. The reference outputs were produced on 2 × V100.

### Expectation value with multi-QPU
`execution=cudaq.parallel.thread` distributes the Hamiltonian terms across the available QPUs.

In [ ]:
cudaq.set_target("nvidia", option="mqpu")
target = cudaq.get_target()
num_qpus = target.num_qpus()
print("Number of QPUs:", num_qpus)


@cudaq.kernel
def ansatz(angle: float):
    qvector = cudaq.qvector(2)
    x(qvector[0])
    ry(angle, qvector[1])
    x.ctrl(qvector[1], qvector[0])


hamiltonian = 5.907 - 2.1433 * spin.x(0) * spin.x(1) - 2.1433 * spin.y(
    0) * spin.y(1) + .21829 * spin.z(0) - 6.125 * spin.z(1)

exp_val = cudaq.observe(ansatz, hamiltonian, 0.59,
                        execution=cudaq.parallel.thread).expectation()
print("Expectation value: ", exp_val)   # reference: -1.7487946640740337

### Parallel circuits
We evaluate $\langle Z_0\rangle$ for **10 000 different parameter sets**:

1. on one GPU, passing the whole parameter array to `cudaq.observe` (it is broadcast over the sets), then
2. with `mqpu`, splitting the sets into 4 batches and sending them asynchronously (`observe_async`) to the available QPUs.

In [ ]:
import time

np.random.seed(1)
cudaq.set_target("nvidia")

qubit_count = 5
sample_count = 10000
h_obs = spin.z(0)
parameter_count = qubit_count

# prepare 10000 different input parameter sets.
parameters = np.random.default_rng(13).uniform(low=0, high=1,
                                               size=(sample_count, parameter_count))


@cudaq.kernel
def rx_kernel(params: list[float]):
    qubits = cudaq.qvector(5)
    for i in range(5):
        rx(params[i], qubits[i])


start_time = time.time()
cudaq.observe(rx_kernel, h_obs, parameters)
print(f"Single GPU, broadcast: {time.time() - start_time:.3f} s")   # reference (V100): 3.75 s

In [ ]:
print('There are', parameters.shape[0], 'parameter sets to execute')

xi = np.split(parameters, 4)  # Split the parameters into 4 arrays
print('Split parameters into', len(xi), 'batches of', xi[0].shape[0], ',',
      xi[1].shape[0], ',', xi[2].shape[0], ',', xi[3].shape[0])

cudaq.set_target("nvidia", option="mqpu")
num_gpus = cudaq.num_available_gpus()
asyncresults = []

start_time = time.time()
for i in range(len(xi)):
    for j in range(xi[i].shape[0]):
        qpu_id = i * num_gpus // len(xi)
        asyncresults.append(
            cudaq.observe_async(rx_kernel, h_obs, xi[i][j, :], qpu_id=qpu_id))
result = [res.get() for res in asyncresults]
print(f"{num_gpus} QPU(s), async batches: {time.time() - start_time:.3f} s")   # reference (2 x V100): 1.60 s

## Next steps
- `02_CUDA_Q_backends.ipynb`: the same GHZ circuit on every simulator backend (state vector, multi-GPU, multi-QPU, tensor network, MPS).
- `03_CUDA_Q_batching.ipynb`: batching large Hamiltonians and parameter sets across QPUs.